In [1]:
%load_ext autoreload
%autoreload 2
%load_ext dotenv
%dotenv

In [21]:
import os
import sys

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"  # see issue #152
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
# os.environ["MKL_THREADING_LAYER"] = "GNU"

sys.path.append("../02-encoding")
sys.path.append(os.path.join(os.environ["NB2P_PREFIX"], "nb2p", "dfgtree"))

In [8]:
from pkgimp import *

from nb2p import fileop, npop, config, stmodel, database

In [9]:
TRAIN_DATASET_NAME = "distilkaggle"
TEST_DATASET_NAME = "distilkaggle"

In [10]:
TRAIN_SETUP_NAME = "astn4_256"
TEST_SETUP_NAME = (
    "astn4_256_cross" if TRAIN_DATASET_NAME != TEST_DATASET_NAME else TRAIN_SETUP_NAME
)
TRAIN_SETUP_NAME, TEST_SETUP_NAME

('astn4_256', 'astn4_256')

In [11]:
TRAIN_DIRS = config.dirs(dataset_name=TRAIN_DATASET_NAME)
TRAIN_DIRS.makedirs()
TEST_DIRS = config.dirs(dataset_name=TEST_DATASET_NAME)
TEST_DIRS.makedirs()

making dirs: /ssd/haotian/scs/distilkaggle/processed-unixcoder-ast
making dirs: /ssd/haotian/scs/distilkaggle
making dirs: /ssd/haotian/scs/distilkaggle/processed-unixcoder-full
making dirs: /ssd/haotian/scs/distilkaggle/dfgtree
making dirs: /ssd/haotian/scs/distilkaggle/processed-unixcoder-eda
making dirs: /ssd/haotian/scs/distilkaggle/logs/full
making dirs: /ssd/haotian/scs/distilkaggle/models/full
making dirs: /ssd/haotian/scs/distilkaggle/ipynb
making dirs: /ssd/haotian/scs/distilkaggle/processed-unixcoder-ast
making dirs: /ssd/haotian/scs/distilkaggle
making dirs: /ssd/haotian/scs/distilkaggle/processed-unixcoder-full
making dirs: /ssd/haotian/scs/distilkaggle/dfgtree
making dirs: /ssd/haotian/scs/distilkaggle/processed-unixcoder-eda
making dirs: /ssd/haotian/scs/distilkaggle/logs/full
making dirs: /ssd/haotian/scs/distilkaggle/models/full
making dirs: /ssd/haotian/scs/distilkaggle/ipynb


## Load Data & Pre-processing

### Load dataset

In [12]:
DEVICE = torch.device("cuda")

In [13]:
MAX_LENGTH = 256
MAX_LENGTH

256

Get the list of all samples by globbing the folder

In [14]:
train_samples = glob.glob(str(TEST_DIRS.dfgtree / "train-*.lz4"))
test_samples = glob.glob(str(TEST_DIRS.dfgtree / "test-*.lz4"))

In [15]:
db, client = database.connect(dataset_name=TRAIN_DATASET_NAME, verbose=True)

Pinged to database nb2p-dk. You successfully connected to MongoDB!


Build List of X-y dicts

In [16]:
ids = set(map(
    lambda x: str(x["_id"]),
    db.notebooksegments.find(
        {"prompted": True, "segment_ends.3": {"$exists": True}, "n_ast_children_of_segments": {"$lte": 256}},
        {"_id": 1},
    ),
))
len(ids)

285874

In [17]:
def clean_segment_ends(segment_ends: List[int]):
    result = set()
    for x in segment_ends:
        if x >= MAX_LENGTH:
            return None, f"segment ends exceed max length: {x}"
        if x not in result and x >= 0:
            result.add(x)

    result = sorted(result)

    if len(result) == 0:
        return None, "empty segment ends after cleaning"

    return result, None

### Build Model Input

In [18]:
def replace_elements(data, replacement_function):
    """
    Recursively replaces elements in an arbitrarily nested list based on a replacement function.

    Args:
        data: The input data, which can be a list or any other iterable.
        replacement_function: A function that takes an element and returns its replacement.

    Returns:
        A new list with replaced elements.
    """

    if not isinstance(data, list):
        return replacement_function(data)

    result = []
    for element in data:
        result.append(replace_elements(element, replacement_function))
    return result

replace_elements([[(0, 1)], [(1, 2), [(2, 3)]]], lambda x: (x[0], x[1] * x[1]))

[[(0, 1)], [(1, 4), [(2, 9)]]]

In [22]:
from nb2p.dfgtree.dfgtree import DFGNode
import queue
from itertools import chain
from collections.abc import Iterable

def flatten(xs):
    for x in xs:
        # print(x)
        if isinstance(x, tuple):
            for node in x[1]:
                yield (x[0], node)
        elif isinstance(x, Iterable) and not isinstance(x, (str, bytes)):
            yield from flatten(x)
        else:
            yield x

def bfs(root: DFGNode):
  """Performs Breadth-First Search on a DFGNode tree.

  Args:
    root: The root node of the DFGNode tree.

  Yields:
    A tuple containing the node's representation and its depth from the root.
  """

  q = queue.Queue()
  q.put((root, 0))

  while not q.empty():
    node, depth = q.get()
    yield node.repr, depth

    for child in node.children:
      q.put((child, depth + 1))

def bfs_layered(root: DFGNode):
    """Performs Breadth-First Search on a DFGNode tree, returning only the layer representations.

    Args:
        root: The root node of the DFGNode tree.

    Returns:
        A list of layers, where each layer is a list of node representations at that depth.
    """

    layers = []
    current_layer = [(0, [root])]
    next_layer = []

    while current_layer:
        nodes = list(flatten([current_layer]))
        if len(nodes) == 0:
            break

        layers.append(current_layer)

        for i, (last_layer_i, node) in enumerate(nodes):
            if node.children:
                next_layer.append((i, node.children))

        current_layer = next_layer
        next_layer = []

    return layers


def build_deep_dataset(samples: List[str]):
    for i, sample in tqdm(enumerate(samples), total=len(samples)):
        # read data
        nb_id = sample.split("/")[-1].split(".")[-2].split("-")[-2]
        if nb_id not in ids:
            # print(f"WARN  ignore {sample}. Reason: should not be included")
            continue
        
        sample_dict = fileop.read_lz4(sample)
        sample_dict["segment_ends"], err = clean_segment_ends(
            sample_dict["segment_ends"]
        )
        if err:
            # print(f"WARN  ignore {sample}. Reason: {err}")
            continue

        sample_dict["y"] = npop.indices_to_binary(
            sample_dict["segment_ends"], sample_dict["segment_ends"][-1] + 1
        )
        
        # extract encodings
        ast_children = []
        for s in sample_dict['segments']:
            node = s['repr']
            ast_children.extend(node.children)

        bfs_result = list(bfs_layered(DFGNode(None, ast_children)))
        bfs_result = bfs_result[1:]
        # pprint(bfs_result)

        if len(bfs_result) == 0:
            # print(f"WARN  ignore {sample}. Reason: encodings is empty")
            continue

        result = list(replace_elements(bfs_result, lambda x: [(x[0], torch.Tensor(b.repr).to(DEVICE)) for b in x[1]]))
        # print(result)
        result = [list(itertools.chain.from_iterable(r)) for r in result]

        yield {
            "x": result,
            "y": sample_dict["y"]
        }

TEST_RESULT = list(build_deep_dataset([train_samples[208], train_samples[209]]))
print(TEST_RESULT)

100%|█████████████████████████████████████████████████████████| 2/2 [00:00<00:00, 19.40it/s]


[{'x': [[(0, tensor([[ 9.4532e-01,  7.5682e-01,  1.0616e+00, -6.5870e-02,  3.1204e-02,
         -1.6171e-01,  5.0097e-01,  9.9698e-01, -1.5415e-01,  2.0015e-01,
         -2.4906e-01, -1.1534e+00, -1.4463e+00,  6.4148e-01,  1.5078e+00,
         -1.4832e+00,  2.3389e+00,  1.3486e+00, -9.6707e-02,  3.4430e-01,
         -6.7489e-01,  4.6825e-01, -2.4847e+00, -3.7632e-01,  8.8007e-01,
          1.4135e+00, -1.1254e+00,  3.0173e-01,  3.9741e-01,  7.0081e-01,
          1.0157e+00, -1.1535e-01, -1.1324e-01, -3.9426e-01, -9.2865e-02,
         -6.9692e-01, -7.2681e-01, -8.0282e-02,  4.2824e-01, -8.5499e-01,
          4.3323e-01, -1.8711e-01,  3.7780e+00, -1.1388e+00, -4.9793e-01,
         -5.6997e-01, -6.5400e-01, -8.5133e-01,  3.1779e-01,  9.9978e-01,
          5.6322e-01,  1.5063e+00,  6.3106e-01,  1.5234e+00,  3.9911e-01,
         -1.0625e+00,  3.8729e-01, -1.3332e+00,  4.6465e-01, -1.7505e+00,
         -1.3916e+00, -2.6076e+00, -6.2693e-01,  8.7838e-01, -5.1067e-01,
          1.8702e+00, -1.

In [23]:
train_encodings = list(build_deep_dataset(train_samples))
len(train_encodings), train_encodings[0]

100%|██████████████████████████████████████████████| 234050/234050 [06:40<00:00, 584.86it/s]


(228578,
 {'x': [[(0,
     tensor([[ 9.4532e-01,  7.5682e-01,  1.0616e+00, -6.5870e-02,  3.1204e-02,
              -1.6171e-01,  5.0097e-01,  9.9698e-01, -1.5415e-01,  2.0015e-01,
              -2.4906e-01, -1.1534e+00, -1.4463e+00,  6.4148e-01,  1.5078e+00,
              -1.4832e+00,  2.3389e+00,  1.3486e+00, -9.6707e-02,  3.4430e-01,
              -6.7489e-01,  4.6825e-01, -2.4847e+00, -3.7632e-01,  8.8007e-01,
               1.4135e+00, -1.1254e+00,  3.0173e-01,  3.9741e-01,  7.0081e-01,
               1.0157e+00, -1.1535e-01, -1.1324e-01, -3.9426e-01, -9.2865e-02,
              -6.9692e-01, -7.2681e-01, -8.0282e-02,  4.2824e-01, -8.5499e-01,
               4.3323e-01, -1.8711e-01,  3.7780e+00, -1.1388e+00, -4.9793e-01,
              -5.6997e-01, -6.5400e-01, -8.5133e-01,  3.1779e-01,  9.9978e-01,
               5.6322e-01,  1.5063e+00,  6.3106e-01,  1.5234e+00,  3.9911e-01,
              -1.0625e+00,  3.8729e-01, -1.3332e+00,  4.6465e-01, -1.7505e+00,
              -1.3916e+00, -2.

In [24]:
test_encodings = list(build_deep_dataset(test_samples))
len(test_encodings), test_encodings[0]

100%|████████████████████████████████████████████████| 58513/58513 [01:39<00:00, 590.98it/s]


(57187,
 {'x': [[(0,
     tensor([[ 9.4532e-01,  7.5682e-01,  1.0616e+00, -6.5870e-02,  3.1204e-02,
              -1.6171e-01,  5.0097e-01,  9.9698e-01, -1.5415e-01,  2.0015e-01,
              -2.4906e-01, -1.1534e+00, -1.4463e+00,  6.4148e-01,  1.5078e+00,
              -1.4832e+00,  2.3389e+00,  1.3486e+00, -9.6707e-02,  3.4430e-01,
              -6.7489e-01,  4.6825e-01, -2.4847e+00, -3.7632e-01,  8.8007e-01,
               1.4135e+00, -1.1254e+00,  3.0173e-01,  3.9741e-01,  7.0081e-01,
               1.0157e+00, -1.1535e-01, -1.1324e-01, -3.9426e-01, -9.2865e-02,
              -6.9692e-01, -7.2681e-01, -8.0282e-02,  4.2824e-01, -8.5499e-01,
               4.3323e-01, -1.8711e-01,  3.7780e+00, -1.1388e+00, -4.9793e-01,
              -5.6997e-01, -6.5400e-01, -8.5133e-01,  3.1779e-01,  9.9978e-01,
               5.6322e-01,  1.5063e+00,  6.3106e-01,  1.5234e+00,  3.9911e-01,
              -1.0625e+00,  3.8729e-01, -1.3332e+00,  4.6465e-01, -1.7505e+00,
              -1.3916e+00, -2.6

### Pad to max length

In [25]:
y_train_padded = np.zeros((len(train_encodings), MAX_LENGTH), dtype=np.float32)
y_test_padded = np.zeros((len(test_encodings), MAX_LENGTH), dtype=np.float32)
y_train_padded.shape, y_test_padded.shape

((228578, 256), (57187, 256))

In [26]:
### NOTE: An interesting finding that applying multithreading is as expected on parallelism
### but not multiprocessing, as NumPy can only use single CPU core across processes.
MAX_WORKERS = 32

from functools import partial


def assign_data4pool(dest, data):
    i, arr = data
    if len(dest.shape) == 2:
        dest[i, : arr.shape[0]] = arr
    elif len(dest.shape) == 3:
        dest[i, : arr.shape[0], :] = arr


def mask_data4pool(data):
    return npop.mask(len(data['y']), MAX_LENGTH)

In [27]:
y_train_padded_arr = thread_map(
    partial(assign_data4pool, y_train_padded),
    enumerate(d['y'] for d in train_encodings),
    max_workers=MAX_WORKERS,
)
print(len(y_train_padded_arr))

228578it [00:00, 420984.29it/s]

228578


In [28]:
train_mask = thread_map(mask_data4pool, train_encodings, max_workers=MAX_WORKERS)
len(train_mask), train_mask[0]

100%|███████████████████████████████████████████| 228578/228578 [00:00<00:00, 497331.91it/s]


(228578,
 array([False, False, False, False, False, False, False, False, False,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  Tru

In [29]:
y_test_padded_arr = thread_map(
    partial(assign_data4pool, y_test_padded),
    enumerate(d["y"] for d in test_encodings),
    max_workers=MAX_WORKERS,
)

57187it [00:00, 394702.08it/s]


In [30]:
test_mask = thread_map(mask_data4pool, test_encodings, max_workers=MAX_WORKERS)
print(test_mask[0])

100%|█████████████████████████████████████████████| 57187/57187 [00:00<00:00, 366820.15it/s]

[False False False False False False False False False  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  T

In [31]:
train_lengths = list(map(lambda x: (~x).sum(), train_mask))
test_lengths = list(map(lambda x: (~x).sum(), test_mask))
print(len(train_lengths), len(test_lengths), len(train_lengths) + len(test_lengths))
print(train_lengths[0], test_lengths[0])

228578 57187 285765
9 9


In [32]:
import gc
import ctypes

del gc.garbage[:]
gc.collect()
libc = ctypes.CDLL("libc.so.6")
libc.malloc_trim(0)

1

## NB2P Decoding Model

In [33]:
torch.autograd.set_detect_anomaly(True)

In [34]:
TRAIN_X = [e['x'] for e in train_encodings]
TEST_X = [e['x'] for e in test_encodings]

In [35]:
def seed_everything(seed: int):
    import random, os
    import numpy as np
    import torch
    
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    
seed_everything(42)

### Train

In [36]:
%xdel model
torch.cuda.empty_cache()
gc.collect()
import ctypes

libc = ctypes.CDLL("libc.so.6")
libc.malloc_trim(0)

NameError: name 'model' is not defined


1

In [37]:
import torch
import numpy as np
np.set_printoptions(precision=6, suppress=True)
torch.set_printoptions(precision=6, sci_mode=False)

input_size = MAX_LENGTH
eval_freq = 1
checkpoint_freq = 1

print(
    input_size,
    eval_freq,
    checkpoint_freq,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

256 1 1
cuda


In [38]:
from nb2p.stmodel.dnn import NB2PDecoder, Trainer, DNNInput

MODEL_NAME = "nb2pdecoder"

hidden_size = 512
num_layers = 3
num_internal_layers = 4

model = NB2PDecoder(
    input_size, 
    hidden_size=hidden_size,
    num_layers=num_layers,
    num_internal_layers=num_internal_layers,
    epsilon_scale=1.0,
    device=device,
    setup='bce',
).to(device)

In [ ]:
Trainer(
    model,
    num_epochs=300,
    batch_size=32,
    eval_freq=eval_freq,
    checkpoint_freq=checkpoint_freq,
    learning_rate=1e-4,
    device=device,
    model_write_dir=TEST_DIRS.log,
    report_interval=50,
    window_size=2000,
    model_name=(
        f"{TRAIN_SETUP_NAME}-{MODEL_NAME}_bf_ff{hidden_size}_l{num_layers}_il{num_internal_layers}"
    ),
    setup='dfgtree-bce',
    # start=0,
).train(
    DNNInput(X=TRAIN_X, y=y_train_padded, code=TRAIN_X, mask=train_mask),
    DNNInput(X=TEST_X, y=y_test_padded, code=TEST_X, mask=test_mask),
    # n_try=10,
)